<a href="https://colab.research.google.com/github/Miguithub/NPP_Cripto_5min/blob/Experimentos-112024-012026/Experimentos_BTC_BS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimentos BS
Notebook editable: calidad → análisis individual y por grupos → selección temporal → cuatro LSTM.
Se predice **el valor de t+1 con información disponible al cierre de t** (5 minutos por defecto).
`BTC_yp` no es el precio en USD. Se conserva su significado y escala originales.
No se usa la prueba para seleccionar variables, arquitectura ni épocas.
Las celdas guardan resultados en una carpeta nueva por ejecución; BS.parquet permanece intacto.

## Lectura real del archivo y verificación previa

Archivo leído: [BS.parquet](https://drive.google.com/file/d/1_xVmgzO9noZzzTNKATIFIw2X6sXdQKfb/view).
Revisión del 14 de octubre de 2026: **131.616 filas y 45 columnas**, calendario UTC regular
de cinco minutos, desde 2024-11-01 00:05 hasta 2026-02-01 00:00, sin duplicados ni huecos.
En el primer 65%: cuatro constantes (`BTC_observed`, `BTC_active_trading`, `BTC__CA`, `BTC__CO`),
sin infinitos, con faltantes rolling de hasta 474 filas en SSR/SSR_S. No confundir esos faltantes con ceros.
Hay escalas muy distintas y colas extremas; por eso se usa mediana/IQR y asinh, sin borrar outliers.

**Selección obtenida con los parámetros predeterminados, solo en entrenamiento:**

- **BTC_yp**: `BTC_yp`, `BTC__Arel`, `BTC__dlogV`, `BTC__SG`, `BTC__rolling_volatility`, `BTC__SP`, `BTC_range_rel_proxy`, `BTC_activity_trades_proxy`, `BTC_v`, `BTC__BILST`, `BTC__PERS_C`, `BTC_taker_sell_usdt_proxy`.

- **BTC__return_k1**: `BTC__return_k1`.

**Mejora media de MAE por subgrupo en los folds internos de train** (positivo = mejora frente al historial del objetivo; cada método tiene su propio baseline):

| Objetivo | Grupo | Ridge | ExtraTrees |
|---|---|---:|---:|
| BTC_yp | actividad | 3.40% | 4.89% |
| BTC_yp | estres | 1.45% | 2.43% |
| BTC_yp | flujo | 2.64% | 3.37% |
| BTC_yp | liquidez | 1.87% | 3.26% |
| BTC_yp | precio_volumen | 4.41% | 3.87% |
| BTC_yp | sistema | -2.25% | 0.87% |
| BTC_yp | todas | 5.31% | 4.97% |
| BTC__return_k1 | actividad | -0.01% | -0.02% |
| BTC__return_k1 | estres | -0.09% | -0.08% |
| BTC__return_k1 | flujo | -0.08% | -0.05% |
| BTC__return_k1 | liquidez | -0.01% | -0.03% |
| BTC__return_k1 | precio_volumen | -0.08% | -0.05% |
| BTC__return_k1 | sistema | -0.09% | -0.08% |
| BTC__return_k1 | todas | -0.33% | -0.04% |

La notebook recalcula esa selección si cambian los datos o los controles. Dentro de cada par se conserva
exactamente la misma lista de variables. Una selección corta significa que las demás no superaron
los filtros de mejora y estabilidad; no se agregan variables para aparentar complejidad.

**Pruebas realizadas:** lectura real, diagnósticos, selección con folds temporales, alineación de etiquetas,
límites de particiones y actualización de pesos por arquitectura, con guardado y recarga.
Estas pruebas comprueban funcionamiento, **no son entrenamiento completo ni evidencian cuál red gana**.
La preferencia previa se explica en la sección 7; la ganadora empírica se decide en la sección 9.

| Red | Parámetros en la prueba |
|---|---:|
| YP_compacta | 12,513 |
| YP_apilada | 32,673 |
| RET_regularizada | 1,297 |
| RET_conv_LSTM | 6,913 |


## 1. Dependencias y Drive
En Colab, ejecutar la siguiente celda. El entrenamiento completo puede requerir GPU.
El notebook no depende de otros scripts del proyecto.

In [ ]:
%pip install -q "tensorflow>=2.16,<2.22" pyarrow pandas numpy scipy matplotlib scikit-learn statsmodels


In [ ]:
try:
    from google.colab import drive
except ImportError:
    print('Entorno local: no se monta Google Drive.')
else:
    drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Panel de edición manual
Cambiar aquí rutas, horizonte, ventana, grupos, selección y entrenamiento.
`ENTRENAR=True` ejecuta las cuatro redes. El control `EVALUAR_TEST` de la última celda
mantiene sellada la prueba hasta terminar la comparación de validación. Para la evaluación
final, cambiar ese control allí y ejecutar solo esa celda en la misma sesión.
No volver a ajustar después de mirar la prueba.
Las variables se suponen disponibles al cierre de la barra. Si una variable se publica
con retraso, declararlo en RETRASOS; no se puede verificar su causalidad solo con el Parquet.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, gc, warnings, hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from IPython.display import display
except ImportError:
    display = print

CARPETA_BS = Path('/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/4. CD/BTC/BS')
ARCHIVO = CARPETA_BS / 'BS.parquet'
OBJETIVOS = ['BTC_yp', 'BTC__return_k1']
HORIZONTE = 1                  # barras hacia adelante, no desplazamiento contemporáneo
VENTANA = 72                   # 6 horas; igual para las dos redes de cada objetivo
FRACCION_TRAIN = 0.65
FRACCION_VAL = 0.20
FRECUENCIA = pd.Timedelta('5min')
LOTE_LECTURA = 8192
MAX_RAM_DF_MB = 512            # límite aproximado, no incluye TensorFlow ni figuras
N_GRAFICOS = 12000             # muestreo solo para figuras
FOLDS = 3
MAX_FILAS_MI = 6000
MAX_FILAS_CV = 16000           # submuestreo uniforme solo para ajustar Ridge/MI
MAX_FEATURES = 12              # incluye historia del propio objetivo
MAX_FALTANTES = 0.10
CORR_REDUNDANCIA = 0.98
MIN_MEJORA_GRUPO = 0.0         # mejora media sobre Ridge autorregresivo
MIN_MEJORA_INDIVIDUAL = 0.0
RETRASOS = {}                 # ejemplo: {'YS': 1}; solo retardos >= 0
EXCLUIR_MANUAL = []
FEATURES_MANUALES = {}        # ejemplo {'BTC_yp': ['BTC_yp','YS_growth']}; mismas X en el par
ENTRENAR = True
SEMILLAS = [42, 123, 2026]    # tres repeticiones: no elegir la mejor semilla
EPOCAS = 35
PACIENCIA = 6
BATCH = 256
LEARNING_RATE = 1e-3
GRAFICOS = True
MOSTRAR_FIGURAS = False        # evita llenar la notebook; archivos PNG siempre disponibles
EJECUCION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SALIDA = CARPETA_BS / ('Experimentos_BS_resultados_' + EJECUCION)

# Cuatro arquitecturas diferentes; estos parámetros son editables.
ARQUITECTURAS = {
    'BTC_yp': {
        'YP_compacta': {'tipo': 'simple', 'unidades': 48, 'dropout': 0.20},
        'YP_apilada': {'tipo': 'stacked', 'unidades': 64, 'segunda': 32, 'dropout': 0.25},
    },
    'BTC__return_k1': {
        'RET_regularizada': {'tipo': 'regularized', 'unidades': 16, 'dropout': 0.30},
        'RET_conv_LSTM': {'tipo': 'conv', 'unidades': 32, 'filtros': 16, 'kernel': 5, 'dropout': 0.30},
    },
}
GRUPOS = {
    'sistema': ['YS_growth','YS','YS_volatility','SYS_imbalance'],
    'precio_volumen': ['BTC_y','BTC_v','BTC_yp','BTC_turnover_usdt_proxy','BTC_range_rel_proxy',
                       'BTC__return_k1','BTC__rolling_volatility','BTC__dlogV','BTC__VSTAB'],
    'actividad': ['BTC_activity_trades_proxy','BTC__TI','BTC__PERS_C','BTC__Arel','BTC__CA','BTC__CO'],
    'flujo': ['BTC_taker_buy_usdt_proxy','BTC_taker_sell_usdt_proxy','BTC__SI','BTC__SG','BTC__SP',
              'BTC__imbalance','BTC__sell_share','BTC__LR_BS','BTC__BIL','BTC__BILST'],
    'liquidez': ['BTC_illiquidity_proxy','BTC__ABS','BTC__ABS_S','BTC__RLIQ','BTC__RLIQ_S','BTC__LIQ'],
    'estres': ['BTC__IR','BTC__STRESS','BTC__SSR','BTC__SSR_S','BTC__AST_S'],
}


## 3. Lectura proyectada con PyArrow y partición temporal
`df` contiene únicamente las columnas numéricas requeridas y bar_end. Su tamaño en este
archivo es pequeño (~23,4 MB nativos; ~47,4 MB en float64 para diagnóstico),
pero se comprueba el presupuesto antes de leer.
No se hace shuffle, no se compactan filas con nulos y no se aceptan huecos ni duplicados.
Cada etiqueta pertenece por completo a su partición; se purgan HORIZONTE orígenes en los bordes.
Las ventanas de validación/prueba pueden usar historia anterior al borde, disponible al pronosticar.

In [ ]:
def cargar_df(path):
    with pq.ParquetFile(path, pre_buffer=False) as pf:
        schema = pf.schema_arrow
        numeric = [f.name for f in schema if pa.types.is_integer(f.type) or pa.types.is_floating(f.type)]
        columns = ['bar_end'] + [c for c in numeric if c.startswith('BTC_') or c.startswith('YS') or c == 'SYS_imbalance']
        missing = set(['bar_end', *OBJETIVOS]) - set(schema.names)
        if missing:
            raise ValueError(f'Faltan columnas: {missing}')
        estimate = pf.metadata.num_rows * len(columns) * 8 * 3 / 1e6
        if estimate > MAX_RAM_DF_MB:
            raise MemoryError(f'Estimación con copias {estimate:.1f} MB > {MAX_RAM_DF_MB} MB; reducir columnas.')
        # Preasignación: no se concatena una lista de dataframes ni ventanas 3D.
        arrays = {c: np.empty(pf.metadata.num_rows, dtype=np.float64) for c in columns if c != 'bar_end'}
        time = np.empty(pf.metadata.num_rows, dtype='datetime64[ns]')
        offset = 0
        for batch in pf.iter_batches(columns=columns, batch_size=LOTE_LECTURA, use_threads=False):
            end = offset + batch.num_rows
            for c in arrays:
                arrays[c][offset:end] = batch.column(c).to_numpy(zero_copy_only=False)
            time[offset:end] = batch.column('bar_end').to_numpy(zero_copy_only=False)
            offset = end
    data = pd.DataFrame(arrays, copy=False)
    data.insert(0, 'bar_end', pd.to_datetime(time, utc=True))
    return data

assert HORIZONTE >= 1 and VENTANA >= 2
assert 0 < FRACCION_TRAIN < 1 and 0 < FRACCION_VAL < 1 - FRACCION_TRAIN
df = cargar_df(ARCHIVO)
if df.bar_end.isna().any() or df.bar_end.duplicated().any() or not df.bar_end.is_monotonic_increasing:
    raise ValueError('Fechas inválidas, repetidas o desordenadas: revisar fuente, no ordenar silenciosamente.')
if not df.bar_end.diff().iloc[1:].eq(FRECUENCIA).all():
    raise ValueError('Hay huecos/cambios de frecuencia: regularizar explícitamente antes de crear ventanas.')
n = len(df)
b1, b2 = int(n * FRACCION_TRAIN), int(n * (FRACCION_TRAIN + FRACCION_VAL))
TRAIN_END = b1 - HORIZONTE
if TRAIN_END < 10 * VENTANA:
    raise ValueError('Entrenamiento demasiado corto para la ventana configurada.')
SALIDA.mkdir(parents=True, exist_ok=False)
Xraw = df.drop(columns='bar_end').replace([np.inf, -np.inf], np.nan)
for c, lag in RETRASOS.items():
    if c not in Xraw or not isinstance(lag, int) or lag < 0:
        raise ValueError(f'Retraso inválido: {c}={lag}')
    Xraw[c] = Xraw[c].shift(lag)
split_table = pd.DataFrame([
    ['train',0,TRAIN_END-1,b1-1], ['validacion',b1,b2-HORIZONTE-1,b2-1],
    ['prueba',b2,n-HORIZONTE-1,n-1]], columns=['particion','origen_desde','origen_hasta','ultima_etiqueta'])
display(split_table)
print('df:', df.shape, '| MB:', round(df.memory_usage(deep=True).sum()/1e6, 2))
print('Cobertura:', df.bar_end.iloc[0], '→', df.bar_end.iloc[-1])
split_table.to_csv(SALIDA/'particiones.csv', index=False)


,particion,origen_desde,origen_hasta,ultima_etiqueta
0,train,0,85548,85549
1,validacion,85550,111871,111872
2,prueba,111873,131614,131615


df: (131616, 45) | MB: 47.38
Cobertura: 2024-11-01 00:05:00+00:00 → 2026-02-01 00:00:00+00:00


## 4. Calidad individual, distribuciones y diagnóstico temporal (solo train)
Constantes, nulos, infinitos, cuantiles, asimetría, colas y autocorrelación.
No se eliminan outliers automáticamente. Las diferencias respetan filas consecutivas.
ADF/KPSS y ARCH son diagnósticos exploratorios; no prueban causalidad ni rentabilidad.
Se usa un tramo continuo de hasta 10.000 barras para esos tests, informado en el reporte.

In [ ]:
def diagnosticar(data):
    rows = []
    for c in data:
        s = data[c].replace([np.inf,-np.inf],np.nan)
        v = s.dropna()
        q = v.quantile([.01,.25,.5,.75,.99])
        rows.append(dict(variable=c, faltantes=float(s.isna().mean()), infinitos=int(np.isinf(data[c]).sum()),
                         unicos=int(v.nunique()), media=v.mean(), std=v.std(), minimo=v.min(), maximo=v.max(),
                         p01=q.get(.01), mediana=q.get(.5), p99=q.get(.99),
                         asimetria=v.skew(), curtosis=v.kurt(), acf1=s.autocorr(1) if v.nunique()>1 else np.nan))
    return pd.DataFrame(rows).set_index('variable')

calidad = diagnosticar(df.drop(columns='bar_end').iloc[:TRAIN_END])
calidad.to_csv(SALIDA/'calidad_train.csv')
display(calidad)
# Estabilidad descriptiva entre tres tramos del entrenamiento, sin inspeccionar test.
estabilidad=[]
for bloque,ids in enumerate(np.array_split(np.arange(TRAIN_END),3),1):
    for c in Xraw:
        s=Xraw[c].iloc[ids]
        estabilidad.append(dict(bloque=bloque,variable=c,mediana=s.median(),
                                iqr=s.quantile(.75)-s.quantile(.25),faltantes=s.isna().mean()))
pd.DataFrame(estabilidad).to_csv(SALIDA/'estabilidad_train.csv',index=False)
excluidas = {}
for c in Xraw:
    s = Xraw[c].iloc[:TRAIN_END]
    if s.isna().mean() > MAX_FALTANTES:
        excluidas[c] = 'exceso de faltantes en train'
    elif s.nunique(dropna=True) <= 1:
        excluidas[c] = 'constante en train'
    elif c.endswith('_AUDIT') or re.search(r'target|lead|future|tplus', c, re.I):
        excluidas[c] = 'auditoría o posible etiqueta futura'
    elif c in EXCLUIR_MANUAL:
        excluidas[c] = 'exclusión manual'
CANDIDATAS = [c for c in Xraw if c not in excluidas]
for target in OBJETIVOS:
    if target not in CANDIDATAS:
        raise ValueError(f'Objetivo no utilizable: {target}; {excluidas.get(target)}')
display(pd.Series(excluidas, name='motivo'))
(SALIDA/'exclusiones.json').write_text(json.dumps(excluidas,indent=2),encoding='utf-8')

def guardar_fig(fig, name):
    fig.tight_layout()
    fig.savefig(SALIDA / (name+'.png'), dpi=120)
    if MOSTRAR_FIGURAS:
        display(fig)
    plt.close(fig)

if GRAFICOS:
    rng = np.random.default_rng(42)
    sample_rows = np.sort(rng.choice(TRAIN_END, min(TRAIN_END,N_GRAFICOS), replace=False))
    for c in Xraw:
        s = Xraw[c].iloc[:TRAIN_END]
        vals = s.iloc[sample_rows].dropna().to_numpy()
        delta = s.diff().iloc[sample_rows].dropna().to_numpy()
        fig, axes = plt.subplots(2,2,figsize=(11,6))
        if len(vals):
            axes[0,0].hist(vals,bins=60,density=True)
            axes[0,1].plot(df.bar_end.iloc[sample_rows],s.iloc[sample_rows],lw=.5)
            locator=mdates.AutoDateLocator(minticks=3,maxticks=5)
            axes[0,1].xaxis.set_major_locator(locator)
            axes[0,1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        if len(vals)>2 and np.ptp(vals)>0:
            stats.probplot(vals,plot=axes[1,0])
        if len(delta):
            axes[1,1].hist(delta,bins=60,density=True)
        for a,title in zip(axes.flat,['Distribución','Serie (muestra)','Q-Q normal','Diferencias consecutivas']):
            a.set_title(title)
        fig.suptitle(f'{c} | solo train | muestra {len(sample_rows):,}')
        guardar_fig(fig,'distribucion_'+c)

def tramo_continuo(s, max_rows=10000):
    # Mayor tramo sin nulos, sin juntar observaciones separadas por huecos.
    valid = s.notna()
    blocks = valid.ne(valid.shift()).cumsum()
    lengths = s[valid].groupby(blocks[valid]).size()
    if lengths.empty:
        return np.array([])
    return s[blocks.eq(lengths.idxmax())].iloc[-max_rows:].to_numpy()

def tests_temporales():
    from statsmodels.tsa.stattools import adfuller, kpss, acf
    from statsmodels.stats.diagnostic import het_arch
    report = []
    for c in CANDIDATAS:
        a = tramo_continuo(Xraw[c].iloc[:TRAIN_END])
        row = {'variable':c, 'n_tramo':len(a)}
        if len(a) >= 200 and np.std(a)>0:
            z = (a-a.mean())/a.std()
            for name, func in {
                'adf_p':lambda:adfuller(z,maxlag=24,autolag='AIC')[1],
                'kpss_p':lambda:kpss(z,regression='c',nlags='auto')[1],
                'arch_p':lambda:het_arch(np.diff(z),nlags=12)[1],
            }.items():
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        row[name] = func()
                except (ValueError,np.linalg.LinAlgError) as e:
                    row[name+'_error'] = str(e)
            if c in OBJETIVOS and GRAFICOS:
                from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
                fig, ax = plt.subplots(1,2,figsize=(11,3))
                plot_acf(z,lags=48,ax=ax[0]); plot_pacf(z,lags=48,ax=ax[1],method='ywm')
                fig.suptitle(c+' | tramo continuo train')
                guardar_fig(fig,'acf_pacf_'+c)
        report.append(row)
    return pd.DataFrame(report)

temporal = tests_temporales()
temporal.to_csv(SALIDA/'diagnosticos_temporales_train.csv',index=False)
display(temporal)


,faltantes,infinitos,unicos,media,std,minimo,maximo,p01,mediana,p99,asimetria,curtosis,acf1
variable,,,,,,,,,,,,,
YS_growth,0.000012,0,85493,4.187970e-06,3.816383e-01,-1.886099e+00,2.923294e+00,-8.852089e-01,-2.070910e-02,1.069467e+00,0.493161,2.020949,-0.324223
YS,0.000000,0,85433,4.960611e+07,5.203695e+07,3.424687e+06,1.376932e+09,8.123651e+06,3.490539e+07,2.454868e+08,5.874771,70.149726,0.816334
YS_volatility,0.000842,0,84899,3.778314e-01,5.722940e-02,1.928558e-01,5.512010e-01,2.386877e-01,3.848300e-01,4.867131e-01,-0.494809,-0.158661,0.999518
SYS_imbalance,0.000000,0,85508,-1.066979e-02,1.687667e-01,-8.402497e-01,8.013449e-01,-4.123731e-01,-9.458195e-03,3.990877e-01,0.015213,0.490832,0.190452
BTC_y,0.000000,0,85447,8.521884e+06,1.217051e+07,9.678484e+04,5.149494e+08,6.695547e+05,5.006172e+06,5.630345e+07,7.823602,137.120286,0.718776
BTC_v,0.000000,0,85269,8.875342e+01,1.300680e+02,9.095800e-01,5.471439e+03,6.545457e+00,5.081882e+01,5.988722e+02,7.646764,129.176242,0.730304
BTC_yp,0.000000,0,85377,1.648985e-01,8.948196e-02,6.275786e-03,7.747708e-01,3.666527e-02,1.465424e-01,4.486035e-01,1.210247,2.077832,0.607033
BTC_observed,0.000000,0,1,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000,0.000000,NaN
BTC_active_trading,0.000000,0,1,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000,0.000000,NaN


,motivo
BTC_observed,constante en train
BTC_active_trading,constante en train
BTC__CA,constante en train
BTC__CO,constante en train
BTC__FLOW_TOTAL_AUDIT,auditoría o posible etiqueta futura
BTC__TURNOVER_PER_TRADE_AUDIT,auditoría o posible etiqueta futura


,variable,n_tramo,adf_p,kpss_p,arch_p
0,YS_growth,10000,0.000000e+00,0.100000,0.000000e+00
1,YS,10000,3.451405e-21,0.059965,1.187933e-106
2,YS_volatility,10000,4.187590e-02,0.010000,2.958320e-03
3,SYS_imbalance,10000,0.000000e+00,0.100000,5.890089e-177
4,BTC_y,10000,8.745532e-25,0.100000,0.000000e+00
5,BTC_v,10000,7.042002e-25,0.100000,0.000000e+00
6,BTC_yp,10000,6.733415e-20,0.100000,1.121567e-188
7,BTC_turnover_usdt_proxy,10000,8.980291e-24,0.100000,0.000000e+00
8,BTC_range_rel_proxy,10000,9.944937e-21,0.100000,0.000000e+00
9,BTC_illiquidity_proxy,10000,2.483767e-24,0.100000,0.000000e+00


## 5. Preprocesamiento ajustado solo con pasado
Imputación por mediana de train, escalado robusto por IQR y asinh para contener colas
sin recortar observaciones. Se vuelve a ajustar dentro de cada fold. Etiquetas nunca imputadas.
Los diagnósticos conservan float64; solo las matrices de entrada de las redes usan float32.

In [ ]:
class Preparador:
    def fit(self, frame):
        a = np.asarray(frame,dtype=float)
        self.median = np.nanmedian(a,axis=0)
        if not np.isfinite(self.median).all():
            raise ValueError('Alguna variable no tiene valores finitos en este entrenamiento.')
        q = np.nanpercentile(a,[25,75],axis=0)
        self.scale = q[1]-q[0]
        fallback = np.nanstd(a,axis=0)
        self.scale = np.where(self.scale>0,self.scale,np.where(fallback>0,fallback,1.0))
        return self
    def transform(self, frame):
        a = np.asarray(frame,dtype=float)
        a = np.where(np.isfinite(a),a,self.median)
        result = np.arcsinh((a-self.median)/self.scale)
        if not np.isfinite(result).all():
            raise ValueError('Transformación produjo valores no finitos.')
        return result.astype('float32')

def subindices(idx, limit):
    return idx[np.linspace(0,len(idx)-1,min(limit,len(idx)),dtype=int)]

def metricas(y,p):
    y,p=np.asarray(y),np.asarray(p)
    return dict(MAE=float(mean_absolute_error(y,p)),RMSE=float(np.sqrt(mean_squared_error(y,p))),
                R2=float(r2_score(y,p)))


## 6. Features individuales y subgrupos contra y(t+h)
Tres folds expansivos dentro de train, con embargo de HORIZONTE barras.
Se informa Pearson, Spearman y MI (dependencia no lineal), estabilidad del signo y
mejora de MAE al agregar cada feature al historial del objetivo. Los subgrupos se
comparan con Ridge y ExtraTrees autorregresivos sobre las mismas fechas.
ExtraTrees contrasta relaciones no lineales; la importancia individual se mide perturbando
cada feature mediante desplazamientos de bloques en el fold futuro. Es importancia predictiva,
no causal. Se incluye el grupo completo y se exige mejora en al menos dos folds.
Estos resultados guían selección y son exploratorios, no estimaciones finales sin sesgo.

In [ ]:
def evaluar_features(target):
    y = df[target].shift(-HORIZONTE).to_numpy()[:TRAIN_END]
    frame = Xraw[CANDIDATAS].iloc[:TRAIN_END]
    rows, group_rows = [], []
    groups = {g:[c for c in cols if c in CANDIDATAS] for g,cols in GRUPOS.items()}
    groups['todas'] = list(CANDIDATAS)
    folds = TimeSeriesSplit(n_splits=FOLDS,gap=HORIZONTE)
    core = [target]
    for fold,(tr,va) in enumerate(folds.split(frame),1):
        tr = tr[np.isfinite(y[tr])]
        va = va[np.isfinite(y[va])]
        tr = subindices(tr,MAX_FILAS_CV)
        va = subindices(va,MAX_FILAS_CV)
        if len(tr)<100 or len(va)<100:
            raise ValueError('Fold sin suficientes etiquetas observadas.')
        prep = Preparador().fit(frame.iloc[tr])
        a,b = prep.transform(frame.iloc[tr]),prep.transform(frame.iloc[va])
        mu,scale = y[tr].mean(),y[tr].std()
        if scale<=0:
            raise ValueError('Objetivo constante en fold.')
        yt=(y[tr]-mu)/scale
        core_idx=[CANDIDATAS.index(c) for c in core]
        base=Ridge(alpha=10).fit(a[:,core_idx],yt)
        base_p=base.predict(b[:,core_idx])*scale+mu
        base_mae=mean_absolute_error(y[va],base_p)
        def tree():
            return ExtraTreesRegressor(n_estimators=64,max_depth=6,min_samples_leaf=64,
                                       max_features=1.0,random_state=42,n_jobs=1)
        nonlinear_base=tree().fit(a[:,core_idx],yt)
        nonlinear_base_mae=mean_absolute_error(y[va],nonlinear_base.predict(b[:,core_idx])*scale+mu)
        full_tree=tree().fit(a,yt)
        imp_idx=subindices(np.arange(len(va)),3000)
        perm_x=b[imp_idx].copy()
        original_mae=mean_absolute_error(y[va][imp_idx],full_tree.predict(perm_x)*scale+mu)
        mi_idx=subindices(np.arange(len(tr)),MAX_FILAS_MI)
        mi=mutual_info_regression(a[mi_idx],yt[mi_idx],random_state=42,n_neighbors=5)
        for j,c in enumerate(CANDIDATAS):
            idx=list(dict.fromkeys(core_idx+[j]))
            p=Ridge(alpha=10).fit(a[:,idx],yt).predict(b[:,idx])*scale+mu
            s=pd.Series(frame[c].iloc[tr].to_numpy())
            yy=pd.Series(y[tr])
            permutation=[]
            original_column=perm_x[:,j].copy()
            for fraction in [1/3,2/3]:
                perm_x[:,j]=np.roll(original_column,max(1,int(len(perm_x)*fraction)))
                changed=mean_absolute_error(y[va][imp_idx],full_tree.predict(perm_x)*scale+mu)
                permutation.append((changed-original_mae)/max(original_mae,1e-15))
            perm_x[:,j]=original_column
            rows.append(dict(objetivo=target,fold=fold,variable=c,MI=mi[j],
                             Pearson=s.corr(yy),Spearman=s.corr(yy,method='spearman'),
                             importancia_no_lineal=np.mean(permutation),
                             mejora_MAE=(base_mae-mean_absolute_error(y[va],p))/max(base_mae,1e-15)))
        for g,cols in groups.items():
            idx=[CANDIDATAS.index(c) for c in dict.fromkeys(core+cols)]
            p=Ridge(alpha=10).fit(a[:,idx],yt).predict(b[:,idx])*scale+mu
            nonlinear_p=tree().fit(a[:,idx],yt).predict(b[:,idx])*scale+mu
            group_rows.append(dict(objetivo=target,fold=fold,grupo=g,n_features=len(idx),
                                   MAE=mean_absolute_error(y[va],p),base_MAE=base_mae,
                                   mejora_no_lineal=(nonlinear_base_mae-mean_absolute_error(y[va],nonlinear_p))/max(nonlinear_base_mae,1e-15),
                                   mejora_MAE=(base_mae-mean_absolute_error(y[va],p))/max(base_mae,1e-15)))
    detail=pd.DataFrame(rows)
    rank=detail.groupby('variable').agg(MI=('MI','mean'),Spearman=('Spearman','mean'),
        Pearson=('Pearson','mean'),mejora_media=('mejora_MAE','mean'),
        mejora_std=('mejora_MAE','std'),fraccion_mejora=('mejora_MAE',lambda s:(s>0).mean()),
        importancia_no_lineal=('importancia_no_lineal','mean'),
        estabilidad_no_lineal=('importancia_no_lineal',lambda s:(s>0).mean()),
        estabilidad_signo=('Spearman',lambda s:abs(np.sign(s.fillna(0)).mean())))
    rank['score_lineal']=rank.mejora_media-0.5*rank.mejora_std.fillna(0)
    rank['score']=0.5*rank.score_lineal.rank(pct=True)+0.5*rank.importancia_no_lineal.rank(pct=True)
    rank=rank.sort_values(['score','MI'],ascending=False)
    gd=pd.DataFrame(group_rows)
    gs=gd.groupby('grupo').agg(mejora_media=('mejora_MAE','mean'),
                             mejora_no_lineal=('mejora_no_lineal','mean'),
                             estabilidad_no_lineal=('mejora_no_lineal',lambda s:(s>0).mean()),
                             estabilidad_lineal=('mejora_MAE',lambda s:(s>0).mean()),
                             mejora_std=('mejora_MAE','std'),MAE=('MAE','mean'))
    allowed_groups=[g for g in gs.index if g!='todas' and
        ((gs.loc[g,'mejora_media']>MIN_MEJORA_GRUPO and gs.loc[g,'estabilidad_lineal']>=2/3) or
         (gs.loc[g,'mejora_no_lineal']>MIN_MEJORA_GRUPO and gs.loc[g,'estabilidad_no_lineal']>=2/3))]
    allowed=set(c for g in allowed_groups for c in groups[g])
    selected=[target]
    redundancy=frame.corr(method='spearman').abs()
    reasons={}
    for c,r in rank.iterrows():
        if c==target:
            reasons[c]='historia propia obligatoria'; continue
        if c not in allowed:
            reasons[c]='subgrupo sin mejora temporal'; continue
        linear_ok=r.mejora_media>MIN_MEJORA_INDIVIDUAL and r.fraccion_mejora>=2/3
        nonlinear_ok=r.importancia_no_lineal>MIN_MEJORA_INDIVIDUAL and r.estabilidad_no_lineal>=2/3
        if not (linear_ok or nonlinear_ok):
            reasons[c]='mejora incremental insuficiente/inestable'; continue
        if any(redundancy.loc[c,s]>=CORR_REDUNDANCIA for s in selected):
            reasons[c]='redundancia'; continue
        if len(selected)>=MAX_FEATURES:
            reasons[c]='límite de features'; continue
        selected.append(c); reasons[c]='seleccionada'
    if target in FEATURES_MANUALES:
        selected=list(dict.fromkeys(FEATURES_MANUALES[target]))
        if not selected or set(selected)-set(CANDIDATAS):
            raise ValueError('Selección manual vacía o con columnas no elegibles.')
        if target not in selected:
            raise ValueError('La selección manual debe incluir la historia del objetivo.')
    rank['decision']=[reasons.get(c,'') for c in rank.index]
    rank['seleccionada']=rank.index.isin(selected)
    rank.to_csv(SALIDA/f'{target}_ranking.csv')
    detail.to_csv(SALIDA/f'{target}_individual_folds.csv',index=False)
    gd.to_csv(SALIDA/f'{target}_grupos_folds.csv',index=False)
    gs.to_csv(SALIDA/f'{target}_grupos_resumen.csv')
    print('\nObjetivo:',target,'| selección:',selected)
    display(rank); display(gs)
    if GRAFICOS:
        fig,axes=plt.subplots(1,3,figsize=(14,8))
        for ax,metric in zip(axes,['Pearson','Spearman','MI']):
            rank[metric].sort_values().plot.barh(ax=ax,title=metric)
        fig.suptitle(target+'(t+h) | relaciones individuales en train')
        guardar_fig(fig,target+'_relaciones_individuales')
        fig,ax=plt.subplots(figsize=(10,4))
        rank.score.sort_values().tail(18).plot.barh(ax=ax,title=target+' | rango combinado lineal / no lineal')
        guardar_fig(fig,target+'_ranking')
        fig,ax=plt.subplots(figsize=(8,4))
        gs[['mejora_media','mejora_no_lineal']].plot.bar(ax=ax,title=target+' | mejora por grupo (MAE)')
        ax.axhline(0,color='black'); guardar_fig(fig,target+'_grupos')
        fig,ax=plt.subplots(figsize=(8,6))
        m=frame[selected].corr(method='spearman')
        im=ax.imshow(m,vmin=-1,vmax=1,cmap='coolwarm')
        ax.set_xticks(range(len(selected)),selected,rotation=90)
        ax.set_yticks(range(len(selected)),selected); fig.colorbar(im,ax=ax)
        guardar_fig(fig,target+'_redundancia')
    return selected,rank,gs

SELECCION={}
RANKINGS={}
GRUPOS_RESULTADOS={}
for objetivo in OBJETIVOS:
    SELECCION[objetivo],RANKINGS[objetivo],GRUPOS_RESULTADOS[objetivo]=evaluar_features(objetivo)
(SALIDA/'features_por_objetivo.json').write_text(json.dumps(SELECCION,indent=2),encoding='utf-8')



Objetivo: BTC_yp | selección: ['BTC_yp', 'BTC__Arel', 'BTC__dlogV', 'BTC__SG', 'BTC__rolling_volatility', 'BTC__SP', 'BTC_range_rel_proxy', 'BTC_activity_trades_proxy', 'BTC_v', 'BTC__BILST', 'BTC__PERS_C', 'BTC_taker_sell_usdt_proxy']


,MI,Spearman,Pearson,mejora_media,mejora_std,fraccion_mejora,importancia_no_lineal,estabilidad_no_lineal,estabilidad_signo,score_lineal,score,decision,seleccionada
variable,,,,,,,,,,,,,
BTC__Arel,0.243917,0.611449,0.579034,0.028636,0.006286,1.000000,7.351418e-02,1.000000,1.000000,0.025493,0.986842,seleccionada,True
BTC__dlogV,0.002418,0.058377,0.074328,0.018419,0.003416,1.000000,4.297604e-03,1.000000,1.000000,0.016711,0.960526,seleccionada,True
BTC__SG,0.002078,0.045307,0.058535,0.011584,0.002067,1.000000,1.209894e-03,1.000000,1.000000,0.010550,0.894737,seleccionada,True
BTC__rolling_volatility,0.082011,0.302012,0.246752,0.014249,0.009745,1.000000,8.765318e-04,1.000000,1.000000,0.009376,0.842105,seleccionada,True
BTC__SP,0.070977,0.349753,0.320797,0.006759,0.003708,1.000000,1.998760e-03,1.000000,1.000000,0.004905,0.842105,seleccionada,True
BTC_range_rel_proxy,0.118188,0.440138,0.361347,0.012147,0.005082,1.000000,5.971672e-04,0.666667,1.000000,0.009606,0.828947,seleccionada,True
BTC_activity_trades_proxy,0.090396,0.392315,0.335442,0.011338,0.004608,1.000000,6.774893e-04,0.666667,1.000000,0.009034,0.815789,seleccionada,True
BTC_v,0.136539,0.471604,0.359517,0.004737,0.001616,1.000000,1.336795e-03,0.666667,1.000000,0.003930,0.802632,seleccionada,True
BTC__RLIQ,0.087148,0.392315,0.361345,0.009762,0.005442,1.000000,4.796406e-04,0.666667,1.000000,0.007041,0.776316,redundancia,False


,mejora_media,mejora_no_lineal,estabilidad_no_lineal,estabilidad_lineal,mejora_std,MAE
grupo,,,,,,
actividad,0.034022,0.048870,1.000000,1.0,0.008569,0.053793
estres,0.014498,0.024306,1.000000,1.0,0.007668,0.054875
flujo,0.026419,0.033731,1.000000,1.0,0.005089,0.054196
liquidez,0.018727,0.032568,1.000000,1.0,0.008023,0.054641
precio_volumen,0.044088,0.038663,1.000000,1.0,0.017014,0.053247
sistema,-0.022542,0.008723,0.666667,0.0,0.036444,0.056878
todas,0.053070,0.049682,1.000000,1.0,0.023713,0.052708



Objetivo: BTC__return_k1 | selección: ['BTC__return_k1']


,MI,Spearman,Pearson,mejora_media,mejora_std,fraccion_mejora,importancia_no_lineal,estabilidad_no_lineal,estabilidad_signo,score_lineal,score,decision,seleccionada
variable,,,,,,,,,,,,,
BTC__rolling_volatility,0.052997,0.006132,0.006049,0.000300,0.000407,0.666667,0.000156,0.666667,1.000000,9.622475e-05,0.986842,subgrupo sin mejora temporal,False
BTC_yp,0.028767,0.004061,0.001051,0.000115,0.000291,0.666667,0.000228,0.666667,0.333333,-3.036528e-05,0.802632,subgrupo sin mejora temporal,False
BTC__RLIQ_S,0.095249,0.007156,0.007608,0.000158,0.000210,1.000000,0.000010,0.666667,1.000000,5.251973e-05,0.763158,subgrupo sin mejora temporal,False
BTC_taker_sell_usdt_proxy,0.098449,0.007156,0.036400,0.000222,0.000311,1.000000,-0.000046,0.333333,1.000000,6.691761e-05,0.671053,subgrupo sin mejora temporal,False
BTC__return_k1,0.038623,-0.002441,0.001455,0.000000,0.000000,0.000000,0.000019,0.666667,0.333333,0.000000e+00,0.657895,historia propia obligatoria,True
BTC__LIQ,0.022301,-0.001960,-0.002534,-0.000009,0.000111,0.333333,0.000059,0.666667,0.333333,-6.483445e-05,0.657895,subgrupo sin mejora temporal,False
BTC_activity_trades_proxy,0.113157,0.004235,0.016300,0.000258,0.000412,1.000000,-0.000036,0.333333,1.000000,5.150305e-05,0.644737,subgrupo sin mejora temporal,False
BTC__ABS,0.096206,-0.006018,-0.000687,0.000102,0.000192,0.666667,0.000009,0.666667,0.333333,6.644636e-06,0.631579,subgrupo sin mejora temporal,False
BTC__BIL,0.010163,0.006733,0.000477,-0.000022,0.000035,0.333333,0.000023,0.666667,1.000000,-3.938460e-05,0.631579,subgrupo sin mejora temporal,False


,mejora_media,mejora_no_lineal,estabilidad_no_lineal,estabilidad_lineal,mejora_std,MAE
grupo,,,,,,
actividad,-0.000147,-0.000224,0.333333,0.333333,0.000569,0.000904
estres,-0.000880,-0.000809,0.000000,0.000000,0.000816,0.000905
flujo,-0.000752,-0.000513,0.000000,0.000000,0.000362,0.000904
liquidez,-0.000128,-0.000308,0.333333,0.000000,0.000096,0.000904
precio_volumen,-0.000815,-0.000528,0.000000,0.000000,0.001075,0.000905
sistema,-0.000876,-0.000769,0.000000,0.333333,0.000858,0.000905
todas,-0.003284,-0.000382,0.000000,0.000000,0.000638,0.000907


321

## 7. Qué diferencia las cuatro redes
| Objetivo | Modelo | Diferencia |
|---|---|---|
| BTC_yp | YP_compacta | Una LSTM de 48 unidades; menor varianza y costo |
| BTC_yp | YP_apilada | LSTM 64 → LSTM 32; mayor capacidad para patrones a distintas escalas |
| Retorno | RET_regularizada | LSTM 16 con penalización L2 más fuerte; prioriza resistir ruido |
| Retorno | RET_conv_LSTM | Convolución causal de 5 barras → LSTM 32; busca patrones locales antes de la memoria |
**Mi preferencia previa:** compacta para yp y regularizada para retorno. En el train del archivo
leído, ACF(1)≈0,607 para yp y ≈−0,0067 para retorno; hay colas extremas en varias features.
Esto justifica empezar con menor complejidad, pero NO demuestra que ganen. La recomendación
empírica se calcula por MAE de validación promedio entre semillas, antes de abrir test.
Las dos redes de cada par comparten features, fechas, ventana, transformaciones, pérdida,
semillas y presupuesto de entrenamiento. Ni MI ni correlación garantizan poder predictivo.

In [ ]:
def construir_modelo(shape, cfg):
    import tensorflow as tf
    k=tf.keras
    inp=k.Input(shape=shape)
    kind=cfg['tipo']
    strength=1e-3 if kind=='regularized' else 1e-4
    reg=k.regularizers.l2(strength)
    x=inp
    if kind=='conv':
        x=k.layers.Conv1D(cfg['filtros'],cfg['kernel'],padding='causal',activation='relu',kernel_regularizer=reg)(x)
    if kind=='stacked':
        x=k.layers.LSTM(cfg['unidades'],return_sequences=True,kernel_regularizer=reg)(x)
        x=k.layers.Dropout(cfg['dropout'])(x)
        x=k.layers.LSTM(cfg['segunda'],kernel_regularizer=reg)(x)
    else:
        x=k.layers.LSTM(cfg['unidades'],kernel_regularizer=reg)(x)
    x=k.layers.Dropout(cfg['dropout'])(x)
    x=k.layers.Dense(16 if kind!='regularized' else 8,activation='relu',kernel_regularizer=reg)(x)
    out=k.layers.Dense(1)(x)
    model=k.Model(inp,out)
    model.compile(optimizer=k.optimizers.Adam(LEARNING_RATE,clipnorm=1.0),loss='mse',metrics=['mae'])
    return model


## 8. Ventanas en lotes, sin materializar el tensor 3D completo
La etiqueta de una ventana terminada en t es y[t+HORIZONTE]. Se descartan únicamente
las ventanas con etiqueta no observada; las X faltantes usan la mediana de train.
No hay cache de todas las ventanas. PyDataset prepara un lote por índice y limita la cola a uno.

In [ ]:
def preparar_objetivo(target):
    features=SELECCION[target]
    prep=Preparador().fit(Xraw[features].iloc[:TRAIN_END])
    x=prep.transform(Xraw[features])
    y=df[target].shift(-HORIZONTE).to_numpy(dtype=float)
    good=np.isfinite(y)
    origins=np.arange(n)
    sets={
        'train':origins[(origins>=VENTANA-1)&(origins<TRAIN_END)&good],
        'val':origins[(origins>=max(b1,VENTANA-1))&(origins<b2-HORIZONTE)&good],
        'test':origins[(origins>=max(b2,VENTANA-1))&(origins<n-HORIZONTE)&good],
    }
    if any(len(v)==0 for v in sets.values()):
        raise ValueError('Partición sin ventanas válidas.')
    mu,sd=y[sets['train']].mean(),y[sets['train']].std()
    if not sd>0:
        raise ValueError('Objetivo constante.')
    np.savez(SALIDA/f'{target}_preprocesamiento.npz',mediana=prep.median,escala=prep.scale,y_mu=mu,y_sd=sd)
    return x,y,sets,mu,sd

def dataset(x,y,origins,mu,sd):
    import tensorflow as tf
    class Ventanas(tf.keras.utils.PyDataset):
        def __init__(self):
            super().__init__(workers=1,use_multiprocessing=False,max_queue_size=1)
        def __len__(self):
            return int(np.ceil(len(origins)/BATCH))
        def __getitem__(self,batch_index):
            if batch_index<0 or batch_index>=len(self):
                raise IndexError(batch_index)
            start=batch_index*BATCH
            idx=origins[start:start+BATCH]
            xb=np.stack([x[t-VENTANA+1:t+1] for t in idx]).astype('float32')
            yb=((y[idx]-mu)/sd).astype('float32')[:,None]
            return xb,yb
    return Ventanas()

def predicciones(model,x,y,origins,mu,sd):
    # Un lote por llamada; evita copias gigantes y conserva el orden de orígenes.
    pieces=[]
    batches=dataset(x,y,origins,mu,sd)
    for i in range(len(batches)):
        xb,_=batches[i]
        pieces.append(model(xb,training=False).numpy().ravel())
    return np.concatenate(pieces)*sd+mu


## 9. Entrenar y comparar en validación
Baselines: media histórica de train, persistencia y cero para retornos.
Se reporta MAE, RMSE y R² en escala original; no se usa MAPE para retornos cercanos a cero.
Early stopping se decide en validación. Tres semillas para medir variabilidad.
El ganador de arquitectura se elige por MAE medio; su predicción final es el promedio
de las semillas, no la semilla más favorable. Las métricas por semilla también se guardan.

In [ ]:
RESULTADOS=[]
MODELOS_GUARDADOS={}
GANADORES={}
PRED_VAL={}
if ENTRENAR:
    import tensorflow as tf
    tf.config.experimental.enable_op_determinism()
    for target in OBJETIVOS:
        x,y,sets,mu,sd=preparar_objetivo(target)
        ids=sets['val']
        actual=y[ids]
        baseline={
            'media_train':np.full(len(ids),mu),
            'persistencia':df[target].to_numpy()[ids],
        }
        if target=='BTC__return_k1':
            baseline['cero']=np.zeros(len(ids))
        for name,p in baseline.items():
            valid=np.isfinite(p)
            RESULTADOS.append(dict(objetivo=target,modelo=name,semilla=-1,**metricas(actual[valid],p[valid])))
        PRED_VAL[target]={}
        for name,cfg in ARQUITECTURAS[target].items():
            paths=[]; preds=[]
            for seed in SEMILLAS:
                tf.keras.backend.clear_session()
                tf.keras.utils.set_random_seed(seed)
                model=construir_modelo((VENTANA,x.shape[1]),cfg)
                model.summary()
                history=model.fit(dataset(x,y,sets['train'],mu,sd),
                    validation_data=dataset(x,y,ids,mu,sd),epochs=EPOCAS,verbose=2,shuffle=False,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_mae',patience=PACIENCIA,restore_best_weights=True),
                               tf.keras.callbacks.TerminateOnNaN()])
                if not all(np.isfinite(history.history['loss'])):
                    raise RuntimeError('Entrenamiento divergente; revisar datos y learning rate.')
                p=predicciones(model,x,y,ids,mu,sd)
                if not np.isfinite(p).all():
                    raise RuntimeError('Predicciones no finitas.')
                best_epoch=int(np.argmin(history.history['val_mae']))+1
                RESULTADOS.append(dict(objetivo=target,modelo=name,semilla=seed,epoca=best_epoch,
                                       parametros=model.count_params(),**metricas(actual,p)))
                path=SALIDA/f'{name}_seed{seed}.keras'
                model.save(path); paths.append(str(path)); preds.append(p)
                pd.DataFrame(history.history).to_csv(SALIDA/f'{name}_seed{seed}_historia.csv',index=False)
                if GRAFICOS:
                    fig,ax=plt.subplots(figsize=(7,3))
                    pd.DataFrame(history.history)[['mae','val_mae']].plot(ax=ax,title=f'{name} seed {seed} | escala normalizada')
                    guardar_fig(fig,f'{name}_seed{seed}_aprendizaje')
                del model; gc.collect()
            MODELOS_GUARDADOS[name]=paths
            PRED_VAL[target][name]=np.mean(preds,axis=0)
        result=pd.DataFrame(RESULTADOS)
        summary=result[(result.objetivo==target)&(result.semilla>=0)].groupby('modelo').agg(
            MAE_media=('MAE','mean'),MAE_std=('MAE','std'),RMSE_media=('RMSE','mean'))
        winner=summary.MAE_media.idxmin(); GANADORES[target]=winner
        display(summary)
        best_base=result[(result.objetivo==target)&(result.semilla<0)].sort_values('MAE').iloc[0]
        improvement=1-summary.loc[winner,'MAE_media']/best_base.MAE
        print(f'Recomendación por validación: {winner}. Mejora vs mejor baseline {best_base.modelo}: {improvement:.2%}.')
        if improvement<=0:
            print('Ninguna ventaja sobre baseline: preferir baseline; no afirmar capacidad predictiva superior.')
        pd.DataFrame({'fecha_objetivo':df.bar_end.iloc[ids+HORIZONTE].to_numpy(),
                      'real':actual,**PRED_VAL[target]}).to_csv(SALIDA/f'{target}_validacion_predicciones.csv',index=False)
        if GRAFICOS:
            fig,ax=plt.subplots(figsize=(12,3))
            for name,p in PRED_VAL[target].items():
                ax.plot(df.bar_end.iloc[ids[-1000:]+HORIZONTE],p[-1000:],label=name,alpha=.7)
            ax.plot(df.bar_end.iloc[ids[-1000:]+HORIZONTE],actual[-1000:],label='real',color='black',lw=.8)
            ax.legend(); guardar_fig(fig,target+'_validacion')
        del x; gc.collect()
    pd.DataFrame(RESULTADOS).to_csv(SALIDA/'metricas_validacion.csv',index=False)
else:
    print('Entrenamiento desactivado. Se completaron los diagnósticos y la selección.')


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 12)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 48)             │        11,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,513 (48.88 KB)

 Trainable params: 12,513 (48.88 KB)

 Non-trainable params: 0 (0.00 B)

...

 Trainable params: 6,913 (27.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/35
334/334 - 29s - 86ms/step - loss: 1.0051 - mae: 0.6456 - val_loss: 0.9147 - val_mae: 0.5791
Epoch 2/35
334/334 - 41s - 122ms/step - loss: 1.0035 - mae: 0.6453 - val_loss: 0.9138 - val_mae: 0.5790
Epoch 3/35
334/334 - 24s - 71ms/step - loss: 1.0026 - mae: 0.6453 - val_loss: 0.9133 - val_mae: 0.5791
Epoch 4/35
334/334 - 27s - 82ms/step - loss: 1.0021 - mae: 0.6452 - val_loss: 0.9129 - val_mae: 0.5791
Epoch 5/35
334/334 - 25s - 75ms/step - loss: 1.0017 - mae: 0.6453 - val_loss: 0.9126 - val_mae: 0.5790
Epoch 6/35
334/334 - 25s - 75ms/step - loss: 1.0010 - mae: 0.6451 - val_loss: 0.9124 - val_mae: 0.5790
Epoch 7/35
334/334 - 26s - 77ms/step - loss: 1.0011 - mae: 0.6453 - val_loss: 0.9121 - val_mae: 0.5790
Epoch 8/35
334/334 - 24s - 72ms/step - loss: 1.0007 - mae: 0.6453 - val_loss: 0.9116 - val_mae: 0.5790
Epoch 9/35
334/334 - 26s - 76ms/step - loss: 0.9999 - mae: 0.6451 - val_loss: 0.9113 - val_mae: 0.5790
Epoch 10/35
334/334 - 41s - 122ms/step - loss: 0.9997 - mae: 0.6451 - va

,MAE_media,MAE_std,RMSE_media
modelo,,,
RET_conv_LSTM,0.000859,6.120656e-08,0.001416
RET_regularizada,0.000859,3.167782e-08,0.001416


Recomendación por validación: RET_conv_LSTM. Mejora vs mejor baseline cero: 0.03%.


## 10. Evaluación final sellada y reproducibilidad
Activar EVALUAR_TEST solo cuando se hayan congelado las decisiones anteriores.
Se evalúa únicamente la arquitectura elegida por validación para cada objetivo.
El intervalo de mejora usa bootstrap de bloques de un día (288 barras), para respetar
parte de la dependencia serial. Es condicional a este split y no cubre todo cambio de régimen.
Una buena predicción de retorno no implica rentabilidad después de costos.

In [ ]:
EVALUAR_TEST = False  # Cambiar aquí a True al congelar las decisiones; ejecutar SOLO esta celda.
manifest=dict(archivo=str(ARCHIVO),filas=n,features=SELECCION,objetivos=OBJETIVOS,
              horizonte=HORIZONTE,ventana=VENTANA,b1=b1,b2=b2,semillas=SEMILLAS,
              arquitecturas=ARQUITECTURAS,ganadores=GANADORES,modelos=MODELOS_GUARDADOS,
              retrasos=RETRASOS,epocas=EPOCAS,batch=BATCH,learning_rate=LEARNING_RATE,
              python=__import__('sys').version,numpy=np.__version__,pandas=pd.__version__,pyarrow=pa.__version__)
manifest['seleccion_config']=dict(folds=FOLDS,max_features=MAX_FEATURES,max_faltantes=MAX_FALTANTES,
    correlacion_redundancia=CORR_REDUNDANCIA,min_mejora_grupo=MIN_MEJORA_GRUPO,
    min_mejora_individual=MIN_MEJORA_INDIVIDUAL,max_filas_cv=MAX_FILAS_CV,max_filas_mi=MAX_FILAS_MI,
    grupos=GRUPOS,features_manuales=FEATURES_MANUALES,excluir_manual=EXCLUIR_MANUAL)
if ENTRENAR:
    manifest['tensorflow']=tf.__version__
(SALIDA/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')

def intervalo_mejora(y,p,base,block=288,reps=1000):
    difference=np.abs(y-base)-np.abs(y-p)
    blocks=[difference[i:i+block] for i in range(0,len(y)-block+1,block)]
    if len(blocks)<5:
        return [np.nan,np.nan]
    means=np.array([v.mean() for v in blocks])
    rng=np.random.default_rng(42)
    samples=means[rng.integers(0,len(means),size=(reps,len(means)))].mean(axis=1)
    return np.quantile(samples,[.025,.975]).tolist()

if EVALUAR_TEST:
    if not GANADORES:
        raise ValueError('Primero entrenar y fijar ganadores en validación.')
    final_rows=[]
    for target,winner in GANADORES.items():
        x,y,sets,mu,sd=preparar_objetivo(target)
        ids=sets['test']; preds=[]
        for path in MODELOS_GUARDADOS[winner]:
            tf.keras.backend.clear_session()
            model=tf.keras.models.load_model(path)
            preds.append(predicciones(model,x,y,ids,mu,sd))
            del model
        p=np.mean(preds,axis=0); actual=y[ids]
        base_options={'media_train':np.full(len(ids),mu),'persistencia':df[target].to_numpy()[ids]}
        if target=='BTC__return_k1':
            base_options['cero']=np.zeros(len(ids))
        validation=pd.DataFrame(RESULTADOS)
        base_name=validation[(validation.objetivo==target)&(validation.semilla<0)].sort_values('MAE').iloc[0].modelo
        for name,bp in {winner:p,**base_options}.items():
            valid=np.isfinite(bp)
            row=dict(objetivo=target,modelo=name,**metricas(actual[valid],bp[valid]))
            if target=='BTC__return_k1':
                row['acierto_signo']=float((np.sign(actual[valid])==np.sign(bp[valid])).mean())
            final_rows.append(row)
        base=base_options[base_name]; valid=np.isfinite(base)
        print(target,winner,'IC95% mejora MAE vs',base_name,intervalo_mejora(actual[valid],p[valid],base[valid]))
        pd.DataFrame({'fecha_objetivo':df.bar_end.iloc[ids+HORIZONTE].to_numpy(),'real':actual,
                      'prediccion':p,'baseline':base}).to_csv(SALIDA/f'{target}_test_predicciones.csv',index=False)
        del x; gc.collect()
    display(pd.DataFrame(final_rows))
    pd.DataFrame(final_rows).to_csv(SALIDA/'metricas_test.csv',index=False)
else:
    print('Prueba sellada: no se calcularon métricas de test.')
print('Resultados guardados en:',SALIDA)


Prueba sellada: no se calcularon métricas de test.
Resultados guardados en: /content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/4. CD/BTC/BS/Experimentos_BS_resultados_20261014T122125116363Z


## Referencias técnicas
- [LSTM en Keras](https://keras.io/api/layers/recurrent_layers/lstm/)
- [Validación temporal y gap](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
La selección es predictiva y exploratoria: no constituye una comprobación causal del principio.